In [ ]:
import os; os.makedirs('solarfil', exist_ok=True)
!pip install -q pycocotools 2>&1 | tail -1

In [ ]:
%%writefile solarfil/__init__.py
"""solarfil: instance segmentation of solar filaments in GONG H-alpha images.

Pipeline
--------
1. preprocess  : limb-darkening flattening + contrast channels (preprocess.py)
2. targets     : multi-annotator soft foreground + instance-border maps (annotations.py)
3. model       : U-Net (segmentation_models_pytorch) with 2 outputs: foreground, border (model.py)
4. training    : high-resolution crops, Dice+BCE, EMA, GroupKFold by photo (train.py)
5. inference   : full-disk prediction, dihedral TTA, fold ensemble (inference.py)
6. instances   : threshold -> fragment linking -> border splitting -> hole filling (postprocess.py)
7. evaluation  : exact re-implementation of the official Panoptic Quality metric (metric.py)
"""
__version__ = "2.0.0"

In [ ]:
%%writefile solarfil/annotations.py
"""MAGFiLO COCO annotations -> ground-truth RLEs and soft training targets.

Important dataset facts (verified on the 2026 Kaggle train split):
  * 707 unique photos, 1154 COCO `images`: a photo can be annotated by 1-3 annotators.
    Each annotator copy has its own image id "<annotator>-<photo stem>".
  * The official metric scores the prediction for a photo against EVERY annotator copy
    separately (see metric.py), so the best prediction is a consensus of annotators.
  * Every annotation is a single filled polygon.
"""
from __future__ import annotations

import json
import os
from collections import defaultdict
from dataclasses import dataclass, field

import cv2
import numpy as np
from pycocotools import mask as mask_util

NATIVE = 2048


@dataclass
class Photo:
    stem: str
    file_name: str
    # annotator image id -> list of polygons (flat [x0, y0, x1, y1, ...])
    annotators: dict = field(default_factory=dict)


def load_photos(json_path: str) -> dict[str, Photo]:
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)
    anns = defaultdict(list)
    for a in coco["annotations"]:
        anns[a["image_id"]].append(a)
    photos: dict[str, Photo] = {}
    for im in coco["images"]:
        stem = os.path.splitext(im["file_name"])[0]
        p = photos.setdefault(stem, Photo(stem=stem, file_name=im["file_name"]))
        polys = []
        for a in anns.get(im["id"], []):
            seg = a["segmentation"]
            if isinstance(seg, list):
                polys.extend([s for s in seg if len(s) >= 6])
        p.annotators[im["id"]] = polys
    return photos


def poly_to_rle(poly: list, h: int = NATIVE, w: int = NATIVE) -> dict:
    rles = mask_util.frPyObjects([poly], h, w)
    return mask_util.merge(rles)


def photo_gt_rles(photo: Photo) -> dict[str, list[dict]]:
    """annotator id -> list of COCO RLE dicts (one per filament), exactly like annToMask."""
    return {aid: [poly_to_rle(p) for p in polys] for aid, polys in photo.annotators.items()}


def soft_targets(photo: Photo, size: int, border_px: int = 5) -> tuple[np.ndarray, np.ndarray]:
    """Soft foreground and instance-border targets at (size, size), averaged over annotators.

    foreground: fraction of annotators (and of the downsampled pixel area) covered by a filament.
    border    : pixels where dilated masks of two different filaments of the same annotator meet.
                Teaches the network where touching filaments must be split.
    """
    fg = np.zeros((size, size), np.float32)
    bd = np.zeros((size, size), np.float32)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * border_px + 1, 2 * border_px + 1))
    n_ann = max(len(photo.annotators), 1)
    for polys in photo.annotators.values():
        union = np.zeros((NATIVE, NATIVE), np.uint8)
        count = np.zeros((NATIVE, NATIVE), np.uint8)
        for p in polys:
            m = mask_util.decode(poly_to_rle(p))
            union |= m
            if len(polys) > 1:
                count += cv2.dilate(m, k)
        fg += cv2.resize(union.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
        if len(polys) > 1:
            b = (count >= 2).astype(np.float32)
            bd += cv2.resize(b, (size, size), interpolation=cv2.INTER_AREA)
    return fg / n_ann, np.clip(bd / n_ann, 0, 1)


def gt_frame(photos: dict[str, Photo], stems: list[str] | None = None):
    """Ground truth in the official submission format (annotator-prefixed filament ids)."""
    import pandas as pd

    rows = []
    for stem in stems or list(photos):
        for aid, polys in photos[stem].annotators.items():
            for i, p in enumerate(polys, 1):
                rle = poly_to_rle(p)
                rows.append({"filament_id": f"{aid}_{i}", "segmentation_rle": rle["counts"].decode()})
    return pd.DataFrame(rows)

In [ ]:
%%writefile solarfil/metric.py
"""Exact re-implementation of the official Panoptic Quality metric (self-evaluation notebook).

Official behaviour reproduced here:
  * GT rows are keyed "<annotator>-<stem>_<k>", prediction rows "<stem>_<k>".
  * For every annotator copy of a photo, ALL predictions of that photo are compared with
    that annotator's filaments; TP iff IoU > 0.5 (strict), unmatched preds = FP,
    unmatched GT = FN.  PQ = sum(IoU_TP) / (TP + FP/2 + FN/2), pooled over everything.
  * Photos without GT (e.g. the other half of the test set) are ignored.
IoUs are computed on RLEs with pycocotools (identical numbers, ~100x faster than dense
2048x2048 matmuls).
"""
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass

import numpy as np
from pycocotools import mask as mask_util

SIZE = [2048, 2048]


def _as_rle(r):
    if isinstance(r, dict):
        return r
    return {"size": SIZE, "counts": r.encode() if isinstance(r, str) else r}


@dataclass
class PQStats:
    iou_sum: float = 0.0
    tp: int = 0
    fp: int = 0
    fn: int = 0

    def add(self, o: "PQStats"):
        self.iou_sum += o.iou_sum
        self.tp += o.tp
        self.fp += o.fp
        self.fn += o.fn
        return self

    @property
    def pq(self) -> float:
        d = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.iou_sum / d if d > 0 else 0.0

    @property
    def sq(self) -> float:
        return self.iou_sum / self.tp if self.tp else 0.0

    @property
    def dq(self) -> float:
        d = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.tp / d if d > 0 else 0.0

    def __repr__(self):
        return (f"PQ={self.pq:.4f} (SQ={self.sq:.4f} DQ={self.dq:.4f}) "
                f"TP={self.tp} FP={self.fp} FN={self.fn}")


def image_stats(gt_rles: list, pred_rles: list) -> PQStats:
    """Stats for one annotator copy vs. the predictions of that photo."""
    n_gt, n_pred = len(gt_rles), len(pred_rles)
    if n_gt == 0:
        return PQStats(fp=n_pred)
    if n_pred == 0:
        return PQStats(fn=n_gt)
    g = [_as_rle(r) for r in gt_rles]
    p = [_as_rle(r) for r in pred_rles]
    iou = np.asarray(mask_util.iou(p, g, [0] * n_gt))  # (n_pred, n_gt)
    hit = iou > 0.5
    return PQStats(
        iou_sum=float(iou[hit].sum()),
        tp=int(hit.sum()),
        fp=int((hit.sum(axis=1) == 0).sum()),
        fn=int((hit.sum(axis=0) == 0).sum()),
    )


def photo_stats(gt_by_annotator: dict, pred_rles: list) -> PQStats:
    s = PQStats()
    for rles in gt_by_annotator.values():
        if len(rles):
            s.add(image_stats(rles, pred_rles))
    return s


def score_frames(gt_df, pred_df) -> PQStats:
    """Score two DataFrames in the official submission format."""
    gt = defaultdict(lambda: defaultdict(list))
    for fid, rle in zip(gt_df["filament_id"], gt_df["segmentation_rle"]):
        ann_img = fid.split("_", 1)[0]
        stem = ann_img.split("-", 1)[1]
        gt[stem][ann_img].append(rle)
    pred = defaultdict(list)
    for fid, rle in zip(pred_df["filament_id"], pred_df["segmentation_rle"]):
        pred[fid.split("_", 1)[0]].append(rle)
    total = PQStats()
    for stem, anns in gt.items():
        total.add(photo_stats(anns, pred.get(stem, [])))
    return total

In [ ]:
%%writefile solarfil/preprocess.py
"""Image preprocessing for GONG H-alpha full-disk observations.

Filaments are dark, elongated absorption features on the solar disk. Their contrast
is low and the disk brightness falls off towards the limb (limb darkening) and varies
between the six GONG sites. We therefore build a 3-channel input:

  ch0  global intensity (percentile-normalised)
  ch1  flattened image  = image / smooth background  (removes limb darkening / gradients)
  ch2  local contrast    = CLAHE of the flattened image

All channels are uint8 so they can be cached cheaply.
"""
from __future__ import annotations

import cv2
import numpy as np

NATIVE = 2048


def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def disk_mask(img: np.ndarray) -> np.ndarray:
    """Binary mask of the solar disk (Otsu on a blurred image, largest component, filled)."""
    small = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
    small = cv2.GaussianBlur(small, (0, 0), 3)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(th)
    if n > 1:
        k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        th = (lab == k).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(th)
    if cnts:
        cv2.drawContours(filled, cnts, -1, 255, -1)
    filled = cv2.erode(filled, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    return cv2.resize(filled, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST) > 0


def flatten(img: np.ndarray, mask: np.ndarray, sigma: float = 24.0) -> np.ndarray:
    """Divide by a smooth background estimated inside the disk (normalised convolution)."""
    s = 4  # work at 1/4 resolution for speed; the background is smooth anyway
    h, w = img.shape
    small = cv2.resize(img.astype(np.float32), (w // s, h // s), interpolation=cv2.INTER_AREA)
    m = cv2.resize(mask.astype(np.float32), (w // s, h // s), interpolation=cv2.INTER_AREA)
    # median filter first so dark filaments don't pull the background down
    med = cv2.medianBlur(np.clip(small, 0, 255).astype(np.uint8), 15).astype(np.float32)
    num = cv2.GaussianBlur(med * m, (0, 0), sigma / s * 2)
    den = cv2.GaussianBlur(m, (0, 0), sigma / s * 2)
    bg = num / np.maximum(den, 1e-3)
    bg = cv2.resize(bg, (w, h), interpolation=cv2.INTER_LINEAR)
    ratio = img.astype(np.float32) / np.maximum(bg, 1.0)
    ratio[~mask] = 1.0
    # map ratio 0.6..1.25 -> 0..255
    out = (ratio - 0.6) / (1.25 - 0.6) * 255.0
    return np.clip(out, 0, 255).astype(np.uint8)


def make_input(img: np.ndarray) -> np.ndarray:
    """Return (H, W, 3) uint8 network input at native resolution."""
    mask = disk_mask(img)
    vals = img[mask] if mask.any() else img.ravel()
    lo, hi = np.percentile(vals, [0.5, 99.5])
    ch0 = np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1.0) * 255.0, 0, 255).astype(np.uint8)
    ch1 = flatten(img, mask)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
    ch2 = clahe.apply(ch1)
    ch2[~mask] = 0
    return np.dstack([ch0, ch1, ch2])


def load_input(path: str, size: int) -> np.ndarray:
    """Read an observation and return the 3-channel input resized to (size, size)."""
    x = make_input(read_gray(path))
    if size != x.shape[0]:
        x = cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA)
    return x

In [ ]:
%%writefile solarfil/postprocess.py
"""Probability maps -> non-overlapping filament instances (COCO RLE at 2048x2048).

Panoptic Quality punishes two failure modes that plain "threshold + connected components"
produces on filaments:
  * fragmentation: one long faint filament breaks into several blobs -> every blob is an FP
    and the real filament an FN.            -> fix: link fragments closer than `link_px`.
  * over-merging : two neighbouring filaments touch -> one blob, IoU < 0.5 with both.
                                             -> fix: cut along the predicted border map.
Annotators draw filled polygons, so instance holes are filled.
All parameters are tuned on out-of-fold predictions with the official metric (tune.py).
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import cv2
import numpy as np
from pycocotools import mask as mask_util
from scipy import ndimage as ndi

NATIVE = 2048


@dataclass
class PostCfg:
    thr: float = 0.45          # foreground probability threshold
    close_px: int = 0          # morphological closing radius before labelling
    min_cc: int = 40           # drop tiny specks before linking (native px)
    link_px: int = 8           # fragments closer than this are one filament (native px)
    border_thr: float = 1.01   # >1 disables border splitting
    min_area: int = 250        # final instance area filter (native px)
    min_score: float = 0.50    # mean foreground prob inside instance
    seed_thr: float = 1.01     # <=1: split instances by watershed from high-confidence seeds
    fill_holes: bool = True

    def as_dict(self):
        return asdict(self)


def _disk(r: int):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def upsample(p: np.ndarray, size: int = NATIVE) -> np.ndarray:
    p = np.ascontiguousarray(p, dtype=np.float32)
    if p.shape[0] == size:
        return p
    return cv2.resize(p, (size, size), interpolation=cv2.INTER_LINEAR)


def instances(fg: np.ndarray, border: np.ndarray | None, cfg: PostCfg, want_feats: bool = False):
    """fg/border: float maps in [0,1] at any square resolution. Returns list of uint8 masks (2048²)."""
    fg = upsample(fg)
    binary = (fg > cfg.thr).astype(np.uint8)
    if cfg.close_px > 0:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, _disk(cfg.close_px))
    if not binary.any():
        return ([], []) if want_feats else []

    # 1) remove specks
    n, lab, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= cfg.min_cc
    binary = keep[lab].astype(np.uint8)
    if not binary.any():
        return ([], []) if want_feats else []

    # 2) link nearby fragments, 3) cut along predicted borders
    core = cv2.dilate(binary, _disk(cfg.link_px // 2)) if cfg.link_px > 1 else binary.copy()
    use_border = border is not None and cfg.border_thr <= 1.0
    if use_border:
        cut = upsample(border) > cfg.border_thr
        core[cut] = 0
    if cfg.seed_thr <= 1.0:
        # seeds = linked high-confidence cores; grow them over the foreground along the probability
        seeds = ((fg > cfg.seed_thr) & (core > 0)).astype(np.uint8)
        if cfg.link_px > 1:
            seeds = cv2.dilate(seeds, _disk(cfg.link_px // 2)) & core
        n, markers = cv2.connectedComponents(seeds, connectivity=8)
        from skimage.segmentation import watershed
        lab = watershed(-fg, markers, mask=core > 0).astype(np.int32)
        # components without any seed keep their own label
        n2, lab2 = cv2.connectedComponents(((core > 0) & (lab == 0)).astype(np.uint8), connectivity=8)
        lab[lab2 > 0] = lab2[lab2 > 0] + n
    else:
        n, lab = cv2.connectedComponents(core, connectivity=8)
        lab = lab.astype(np.int32)
    lab[binary == 0] = 0
    if use_border:
        # border pixels that belong to the foreground: give them to the nearest instance
        orphan = (binary > 0) & (lab == 0)
        if orphan.any() and n > 1:
            _, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_indices=True)
            lab[orphan] = lab[iy[orphan], ix[orphan]]

    # 4) per-instance filtering
    out, feats = [], []
    objs = ndi.find_objects(lab)
    n_obj = sum(o is not None for o in objs)
    bd_full = upsample(border) if (border is not None and want_feats) else None
    for i, sl in enumerate(objs, 1):
        if sl is None:
            continue
        m = lab[sl] == i
        if cfg.fill_holes:
            m = ndi.binary_fill_holes(m)
            m &= (lab[sl] == i) | (lab[sl] == 0)  # never steal pixels of another instance
        area = int(m.sum())
        if area < cfg.min_area:
            continue
        pv = fg[sl][m]
        if float(pv.mean()) < cfg.min_score:
            continue
        full = np.zeros((NATIVE, NATIVE), np.uint8)
        full[sl][m] = 1
        out.append(full)
        if want_feats:
            feats.append(_features(m, sl, pv, bd_full[sl][m] if bd_full is not None else None, n_obj))
    return (out, feats) if want_feats else out


FEATURES = ["log_area", "mean_p", "max_p", "p10", "p50", "p90", "std_p", "f50", "f70", "f90",
            "mean_bd", "h", "w", "fill", "elong", "r_center", "n_obj"]


def _features(m, sl, pv, bv, n_obj):
    h = sl[0].stop - sl[0].start
    w = sl[1].stop - sl[1].start
    ys, xs = np.nonzero(m)
    cy, cx = ys.mean() + sl[0].start, xs.mean() + sl[1].start
    if len(ys) > 2:
        ev = np.linalg.eigvalsh(np.cov(np.vstack([ys, xs])))
        elong = float(np.sqrt(max(ev[-1], 1e-6) / max(ev[0], 1e-6)))
    else:
        elong = 1.0
    q = np.percentile(pv, [10, 50, 90])
    return [np.log(m.sum()), float(pv.mean()), float(pv.max()), q[0], q[1], q[2], float(pv.std()),
            float((pv > .5).mean()), float((pv > .7).mean()), float((pv > .9).mean()),
            float(bv.mean()) if bv is not None else 0.0, h, w, m.sum() / max(h * w, 1), min(elong, 50.0),
            float(np.hypot(cy - 1024, cx - 1024) / 1024), n_obj]


def load_avg(paths: list[str]) -> np.ndarray:
    """Average uint8 probability npz files (2,S,S) of possibly different resolutions."""
    ps = [np.load(f)["p"].astype(np.float32) / 255.0 for f in paths]
    top = max(p.shape[-1] for p in ps)
    ps = [p if p.shape[-1] == top else np.stack([upsample(c, top) for c in p]) for p in ps]
    return np.mean(ps, axis=0)


def encode(masks: list[np.ndarray]) -> list[str]:
    return [mask_util.encode(np.asfortranarray(m))["counts"].decode() for m in masks]


def instances_rle(fg, border, cfg: PostCfg, rescorer=None) -> list[str]:
    """rescorer: optional dict {"model": fitted regressor, "thr": float} (see rescore.py)."""
    if rescorer is None:
        return encode(instances(fg, border, cfg))
    masks, feats = instances(fg, border, cfg, want_feats=True)
    if not masks:
        return []
    s = rescorer["model"].predict(np.asarray(feats, np.float32))
    return encode([m for m, v in zip(masks, s) if v >= rescorer["thr"]])

In [ ]:
%%writefile solarfil/visualize.py
"""H-alpha overlays of predicted filament masks (contours on the original image)."""
from __future__ import annotations

import os

import cv2
import numpy as np
import pandas as pd
from pycocotools import mask as mask_util

from .preprocess import read_gray


def decode(rle: str, size: int = 2048) -> np.ndarray:
    """COCO RLE string -> uint8 mask (size x size)."""
    return mask_util.decode({"size": [size, size], "counts": rle.encode()})


def overlay(img: np.ndarray, rles: list[str], gt_rles: list[str] | None = None) -> np.ndarray:
    """Draw prediction contours (yellow, numbered) and optional GT contours (cyan) on a gray image."""
    out = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for rs, col in ((gt_rles or [], (255, 255, 0)), (rles, (0, 220, 255))):
        for i, r in enumerate(rs, 1):
            cnts, _ = cv2.findContours(decode(r), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(out, cnts, -1, col, 2)
            if col == (0, 220, 255) and cnts:
                x, y, _, _ = cv2.boundingRect(cnts[0])
                cv2.putText(out, str(i), (x, max(y - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
    return out


def render_submission(sub_csv: str, img_dir: str, out_dir: str, n: int = 12, size: int = 1024) -> list[str]:
    """Write overlay PNGs for the ``n`` test images with the most predicted filaments."""
    df = pd.read_csv(sub_csv)
    df["stem"] = df["filament_id"].str.split("_", n=1).str[0]
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for stem in df["stem"].value_counts().index[:n]:
        src = next(os.path.join(img_dir, f"{stem}{e}") for e in (".jpg", ".jpeg", ".png")
                   if os.path.exists(os.path.join(img_dir, f"{stem}{e}")))
        vis = overlay(read_gray(src), df.loc[df["stem"] == stem, "segmentation_rle"].tolist())
        path = os.path.join(out_dir, f"{stem}.png")
        cv2.imwrite(path, cv2.resize(vis, (size, size), interpolation=cv2.INTER_AREA))
        paths.append(path)
    return paths

In [ ]:
import glob, json, os, sys
import numpy as np, pandas as pd, cv2
sys.path.insert(0, os.getcwd())
from pycocotools import mask as mu
from solarfil.annotations import load_photos, photo_gt_rles
from solarfil.metric import photo_stats
from solarfil.postprocess import PostCfg, instances_rle, load_avg
from solarfil.preprocess import read_gray
from solarfil.visualize import decode
DATA = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json', recursive=True)[0]))
OUT = '/kaggle/working/figs'; os.makedirs(OUT, exist_ok=True)
cfg = PostCfg(thr=.35, close_px=0, min_cc=40, link_px=14, border_thr=1.01, min_area=300, min_score=.8, seed_thr=1.01, fill_holes=False)
photos = load_photos(f"{DATA}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json")
oofdir = glob.glob('/kaggle/input/**/oof_Unet_efficientnet-b4_s1536', recursive=True)[0]
stems = sorted(os.path.basename(f)[:-4] for f in glob.glob(oofdir + '/*.npz'))
print(len(stems), 'fold-0 OOF photos')
def draw(img, rles, col, th=2):
    for r in rles:
        c, _ = cv2.findContours(decode(r), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(img, c, -1, col, th)
cand = []
for s in stems:
    gts = photo_gt_rles(photos[s]); aids = [a for a, g in gts.items() if g]
    if not aids: continue
    p = load_avg([f"{oofdir}/{s}.npz"]); pr = instances_rle(p[0], p[1], cfg)
    st = photo_stats(gts, pr)
    cand.append((st.pq, len(pr), s, aids[0], pr))
cand = [c for c in cand if c[1] >= 5]
cand.sort(key=lambda c: -c[0])
sel = [cand[0], cand[len(cand)//2]]   # a good and a typical case
for k, (pq, n, s, aid, pr) in enumerate(sel, 1):
    path = glob.glob(f"{DATA}/train/**/{s}.*", recursive=True)
    path = [p for p in path if p.lower().endswith(('.jpg', '.jpeg', '.png'))][0]
    img = cv2.cvtColor(read_gray(path), cv2.COLOR_GRAY2BGR)
    gts = photo_gt_rles(photos[s])
    left, right = img.copy(), img.copy()
    for a, g in gts.items():
        draw(left, [x["counts"].decode() for x in g], (255, 255, 0))
    draw(right, pr, (0, 220, 255))
    both = np.concatenate([cv2.resize(left, (1024, 1024), interpolation=cv2.INTER_AREA), cv2.resize(right, (1024, 1024), interpolation=cv2.INTER_AREA)], 1)
    cv2.imwrite(f"{OUT}/overlay_{k}.png", both)
    print(k, s, 'photo PQ %.3f' % pq, 'n_pred', n)